In [0]:
catalog = '`de-foundation-01`'

In [0]:
from pyspark.sql.functions import (
    coalesce,
    lit,
    when,
    try_to_date,
    col
)

actual = spark.table(
    f"{catalog}.bronze.actual_cost"
)

actual_typed = (
    actual
    .withColumn(
        "Cost_Date_Clean",
        coalesce(
            try_to_date(col("Cost_Date"), "yyyy-MM-dd"),
            try_to_date(col("Cost_Date"), "yyyy/MM/dd")
        )
    )
    .withColumn(
        "Quantity_Clean",
        col("Quantity").cast("decimal(18,2)")
    )
    .withColumn(
        "Amount_Clean",
        col("Amount").cast("decimal(18,2)")
    )
)

In [0]:
project_keys = (
    spark.table(f"{catalog}.silver.project")
    .select("Project_Code")
    .withColumn("_valid_project", lit(1))
)

actual_checked = (
    actual_typed
    .join(
        project_keys,
        on="Project_Code",
        how="left"
    )
)

In [0]:
project_keys = (
    spark.table(f"{catalog}.silver.project")
    .select("Project_Code")
    .withColumn("_valid_project", lit(1))
)

actual_checked = (
    actual_typed
    .join(
        project_keys,
        on="Project_Code",
        how="left"
    )
)

In [0]:
actual_checked = (
    actual_checked
    .withColumn(
        "_validation_error",
        when(
            col("Cost_Date_Clean").isNull(),
            "INVALID_DATE"
        )
        .when(
            col("Amount_Clean").isNull(),
            "INVALID_AMOUNT"
        )
        .when(
            col("_valid_project").isNull(),
            "INVALID_PROJECT"
        )
    )
)

In [0]:
display(
    actual_checked.filter(
        col("_validation_error").isNotNull()
    )
)

In [0]:
quarantine_actual = (
    actual_checked
    .filter(
        col("_validation_error").isNotNull()
    )
)

In [0]:
silver_actual = (
    actual_checked
    .filter(
        col("_validation_error").isNull()
    )
    .dropDuplicates(
        ["Source_Reference"]
    )
    .drop(
        "_valid_project",
        "_validation_error"
    )
)

In [0]:
(
    silver_actual.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{catalog}.silver.actual_cost"
    )
)

(
    quarantine_actual.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{catalog}.silver.actual_cost_quarantine"
    )
)